In [1]:
import pymupdf as fitz  # Updated import for PyMuPDF
import re
from typing import List, Dict, Any

In [2]:
def process_wind_regulations(pdf_path: str, chunk_size: int = 2000) -> List[Dict[str, Any]]:
    """Process wind farm regulations optimized for LLM understanding with minimal tokens"""
    doc = fitz.open(pdf_path)
    chunks = []
    doc_id = pdf_path.split("/")[-1].replace(".pdf", "")
    
    # Extract only critical document-level metadata once
    first_page_text = doc[0].get_text()
    jurisdiction = extract_jurisdiction(first_page_text)
    pub_date = extract_date(first_page_text)
    
    full_text = ""
    page_markers = {}
    
    # First pass: collect text with page markers
    for page_num in range(len(doc)):
        page_text = doc[page_num].get_text()
        marker = f"[pg{page_num+1}]"
        page_markers[marker] = page_num + 1
        full_text += marker + page_text
    
    # Smart chunking at paragraph boundaries
    paragraphs = re.split(r'\n\s*\n', full_text)
    
    current_chunk = ""
    current_pages = set()
    
    for paragraph in paragraphs:
        # Extract page markers
        for marker in page_markers:
            if marker in paragraph:
                current_pages.add(page_markers[marker])
                paragraph = paragraph.replace(marker, "")
        
        # Check if adding this paragraph exceeds chunk size
        if len(current_chunk) + len(paragraph) > chunk_size and current_chunk:
            # Save the current chunk
            chunks.append({
                "text": current_chunk.strip(),
                "doc_id": doc_id,
                "pages": f"{min(current_pages)}-{max(current_pages)}",
                **({"jurisdiction": jurisdiction} if jurisdiction else {}),
                **({"date": pub_date} if pub_date else {})
            })
            
            # Start new chunk
            current_chunk = paragraph
            current_pages = set()
            for marker in page_markers:
                if marker in paragraph:
                    current_pages.add(page_markers[marker])
        else:
            current_chunk += "\n\n" + paragraph if current_chunk else paragraph
    
    # Add the final chunk
    if current_chunk:
        chunks.append({
            "text": current_chunk.strip(),
            "doc_id": doc_id,
            "pages": f"{min(current_pages)}-{max(current_pages)}",
            **({"jurisdiction": jurisdiction} if jurisdiction else {}),
            **({"date": pub_date} if pub_date else {})
        })
    
    return chunks

def extract_jurisdiction(text):
    """Extract jurisdiction information using minimal pattern matching"""
    patterns = [
        r"((?:Federal|State|National|EU|IEC|IEEE|ISO) (?:Regulation|Standard|Directive))"
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    return None

def extract_date(text):
    """Extract publication date using minimal pattern matching"""
    patterns = [
        r"(?:Published|Date):?\s+(\d{1,2}[\/\.\-]\d{1,2}[\/\.\-]\d{2,4})",
        r"((?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4})"
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    return None

In [3]:
import pymupdf4llm
import re
from typing import List, Dict, Any
import json

def extract_jurisdiction(text: str) -> str | None:
    """Extract jurisdiction information using minimal pattern matching."""
    patterns = [
        r"((?:Federal|State|National|EU|IEC|IEEE|ISO) (?:Regulation|Standard|Directive))"
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    return None

def extract_date(text: str) -> str | None:
    """Extract publication date using minimal pattern matching."""
    patterns = [
        r"(?:Published|Date):?\s+(\d{1,2}[\/\.\-]\d{1,2}[\/\.\-]\d{2,4})",
        r"((?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4})"
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    return None

def extract_wind_regulations(pdf_path: str, chunk_size: int = 2000) -> List[Dict[str, Any]]:
    """
    Extract wind farm regulations from a PDF using pymupdf4llm's to_markdown,
    chunk for LLMs, and add minimal metadata.
    """
    # Extract markdown text from PDF
    md_text = pymupdf4llm.to_markdown(pdf_path)
    doc_id = pdf_path.split("/")[-1].replace(".pdf", "")

    # Split markdown into paragraphs for chunking
    paragraphs = re.split(r'\n\s*\n', md_text)
    chunks = []
    current_chunk = ""
    for para in paragraphs:
        if len(current_chunk) + len(para) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            current_chunk += "\n\n" + para if current_chunk else para
    if current_chunk:
        chunks.append(current_chunk.strip())

    # Extract metadata from the first chunk
    jurisdiction = extract_jurisdiction(chunks[0]) if chunks else None
    pub_date = extract_date(chunks[0]) if chunks else None

    # Build output with metadata
    output = []
    for idx, chunk in enumerate(chunks):
        entry = {
            "text": chunk,
            "doc_id": doc_id,
            "chunk": idx + 1,
            **({"jurisdiction": jurisdiction} if jurisdiction else {}),
            **({"date": pub_date} if pub_date else {})
        }
        output.append(entry)
    return output

In [4]:
# Example usage:
pdf_path = "project_folder/LLM/Documents/27.pdf"

chunks = extract_wind_regulations(pdf_path)

with open("27.json", "w") as f:
    json.dump(chunks, f, indent=2)

In [5]:
import os
from pathlib import Path

def process_pdf_folder(folder_path: str, output_folder: str = None, chunk_size: int = 2000):
    """
    Process all PDF files in a folder and save each as a JSON file.
    
    Args:
        folder_path: Path to folder containing PDF files
        output_folder: Path to save JSON files (defaults to same folder)
        chunk_size: Size of text chunks for processing
    """
    folder_path = Path(folder_path)
    if output_folder is None:
        output_folder = folder_path
    else:
        output_folder = Path(output_folder)
        output_folder.mkdir(exist_ok=True)
    
    # Find all PDF files in the folder
    pdf_files = list(folder_path.glob("*.pdf"))
    
    if not pdf_files:
        print(f"No PDF files found in {folder_path}")
        return
    
    print(f"Found {len(pdf_files)} PDF files to process...")
    
    # Process each PDF file
    for pdf_file in pdf_files:
        try:
            print(f"Processing: {pdf_file.name}")
            
            # Extract chunks from PDF
            chunks = extract_wind_regulations(str(pdf_file), chunk_size)
            
            # Create output filename (same name but .json extension)
            json_filename = pdf_file.stem + ".json"
            json_path = output_folder / json_filename
            
            # Save to JSON file
            with open(json_path, "w", encoding='utf-8') as f:
                json.dump(chunks, f, indent=2, ensure_ascii=False)
            
            print(f"  ✓ Saved {len(chunks)} chunks to {json_filename}")
            
        except Exception as e:
            print(f"  ✗ Error processing {pdf_file.name}: {str(e)}")
    
    print("Batch processing complete!")

# Example usage for processing multiple files:
def process_documents_batch():
    """Example function showing how to process multiple PDFs"""
    
    # Specify the folder containing PDF files
    pdf_folder = "project_folder/LLM/Documents"
    
    # Specify output folder (optional - will use same folder if not specified)
    output_folder = "project_folder/LLM/Preprocessing"
    
    # Process all PDFs in the folder
    process_pdf_folder(pdf_folder, output_folder, chunk_size=2000)

# Alternative: Process specific files by pattern
def process_specific_pdfs(folder_path: str, pattern: str = "*.pdf"):
    """Process PDFs matching a specific pattern"""
    folder_path = Path(folder_path)
    
    # Find PDFs matching pattern (e.g., "regulation*.pdf", "wind*.pdf", etc.)
    pdf_files = list(folder_path.glob(pattern))
    
    for pdf_file in pdf_files:
        try:
            print(f"Processing: {pdf_file.name}")
            chunks = extract_wind_regulations(str(pdf_file))
            
            # Save in the same folder as source
            json_path = pdf_file.parent / f"{pdf_file.stem}.json"
            with open(json_path, "w", encoding='utf-8') as f:
                json.dump(chunks, f, indent=2)
                
            print(f"  ✓ Saved to {json_path.name}")
        except Exception as e:
            print(f"  ✗ Error: {e}")

In [6]:
# Run the batch processing
# Uncomment the line below to process all PDFs in your Documents folder

# process_documents_batch()

# Or process PDFs in a specific folder:
pdf_folder = "project_folder/LLM/Documents"
output_folder = "project_folder/LLM/Json_docs"

process_pdf_folder(pdf_folder, output_folder)

Found 49 PDF files to process...
Processing: 49.pdf
  ✓ Saved 361 chunks to 49.json
Processing: 48.pdf
  ✓ Saved 372 chunks to 48.json
Processing: 9.pdf
  ✓ Saved 106 chunks to 9.json
Processing: 8.pdf
  ✓ Saved 69 chunks to 8.json
Processing: 16.pdf
  ✓ Saved 87 chunks to 16.json
Processing: 17.pdf
  ✓ Saved 227 chunks to 17.json
Processing: 29.pdf
  ✓ Saved 254 chunks to 29.json
Processing: 15.pdf
  ✓ Saved 16 chunks to 15.json
Processing: 14.pdf
  ✓ Saved 24 chunks to 14.json
Processing: 28.pdf
  ✓ Saved 1 chunks to 28.json
Processing: 10.pdf
  ✓ Saved 442 chunks to 10.json
Processing: 38.pdf
  ✓ Saved 18 chunks to 38.json
Processing: 39.pdf
  ✓ Saved 299 chunks to 39.json
Processing: 11.pdf
  ✓ Saved 532 chunks to 11.json
Processing: 13.pdf
  ✓ Saved 161 chunks to 13.json
Processing: 12.pdf
  ✓ Saved 161 chunks to 12.json
Processing: 23.pdf
  ✓ Saved 616 chunks to 23.json
Processing: 37.pdf
  ✓ Saved 119 chunks to 37.json
Processing: 36.pdf
  ✓ Saved 32 chunks to 36.json
Processing